# Step 08 — Final Results Aggregation

**Input:**
- `data/output/07_building_level_redistributed.gpkg`
- `data/input/zone_targets.gpkg`

**Output:** `data/output/08_final_results.gpkg`

Aggregates building-level assigned activity counts back to zone level,
merges with official zone targets for comparison, and exports the final dataset.

In [1]:
import sys
sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))
from config import *

import geopandas as gpd
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
print('Config loaded')

Config loaded


## 1. Load redistribution results and zones

In [2]:
buildings = gpd.read_file(REDISTRIBUTION_FILE)
zones     = gpd.read_file(ZONE_TARGETS_FILE, layer=ZONE_LAYER if ZONE_LAYER else None).to_crs(TARGET_CRS)
zones = zones.reset_index(drop=True)
zones['zone_id'] = zones.index

print(f'Buildings: {len(buildings):,}')
print(f'Zones:     {len(zones):,}')

Buildings: 282,017
Zones:     880


## 2. Spatial join buildings to zones

In [3]:
bld_cents = buildings.copy()
bld_cents['geometry'] = buildings.geometry.centroid

bld_with_zone = gpd.sjoin(bld_cents, zones[['zone_id', 'geometry']], how='left', predicate='within')
bld_with_zone = bld_with_zone.drop_duplicates(subset='gml_id')
print(f'Buildings matched to a zone: {bld_with_zone["zone_id"].notna().sum():,}')

Buildings matched to a zone: 279,913


## 3. Aggregate to zone level

In [4]:
assigned_cols = [f'assigned_{act.replace("-", "_")}' for act in ZONE_ACTIVITY_COLUMNS
                 if f'assigned_{act.replace("-", "_")}' in bld_with_zone.columns]

zone_agg = (
    bld_with_zone[bld_with_zone['zone_id'].notna()]
    .groupby('zone_id')[assigned_cols]
    .sum()
    .reset_index()
)

# Merge with zones
zones_result = zones.merge(zone_agg, on='zone_id', how='left')
zones_result[assigned_cols] = zones_result[assigned_cols].fillna(0)

print('Zone-level aggregated results:')
print(zones_result[assigned_cols].describe().round(1))

Zone-level aggregated results:
       assigned_Workers  assigned_School  assigned_University  \
count             880.0            880.0                880.0   
mean              672.4            170.5                 43.5   
std              2637.2            465.9                464.7   
min                 0.0              0.0                  0.0   
25%                16.4              0.0                  0.0   
50%               147.3              0.0                  0.0   
75%               651.9            136.8                  0.0   
max             70309.4           5092.7               8386.1   

       assigned_Kindergarten  assigned_Retail_Daily  \
count                  880.0                  880.0   
mean                    42.4                 1032.8   
std                     61.5                 2241.3   
min                      0.0                    0.0   
25%                      0.0                    0.0   
50%                      0.0                   67.0  

## 4. Save final results

In [5]:
buildings.to_file(FINAL_RESULTS_FILE, driver='GPKG', layer='buildings')
zones_result.to_file(FINAL_RESULTS_FILE, driver='GPKG', layer='zones')

print(f'Saved building-level results and zone-level aggregation → {FINAL_RESULTS_FILE}')
print('\nPipeline complete!')

Saved building-level results and zone-level aggregation → C:\Users\Mayur Patel\Documents\GitHub\Capacity_Calculation-pipeline-optimized\data\output\08_final_results.gpkg

Pipeline complete!
